# Hospital Inpatient Discharges: Exploratory Data Analysis
**Mathematics for AI-II · Summative Assessment** · NY SPARCS 2021

This notebook documents the exploratory analysis behind the dashboard. It loads the cleaned sample produced by `scripts/clean_data.py`, answers the seven research questions with `groupby`/`agg` and `pivot_table`, and runs the statistical tests.

**How to use in Colab:** upload `discharges_clean.parquet` (or mount your repo) and run all cells.

In [ ]:
import pandas as pd, numpy as np, plotly.express as px
from scipy import stats
pd.set_option("display.width", 200); pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_parquet("discharges_clean.parquet")      # change path if needed
SCALE = 5.2535   # full cleaned rows / sample rows (see data/cleaning_report.json)
print(df.shape); df.head()

## 1. Data structure and quality

In [ ]:
print(df.dtypes.to_string())
print("\nMissing values per column:"); print(df.isna().sum()[lambda s: s > 0])
df[["los","total_charges","total_costs","charge_per_day","charge_to_cost_ratio"]].describe(percentiles=[.25,.5,.75,.95,.99]).T

**Reading the summary:** the mean stay (5.75 d) is almost double the median (3 d) and charges show the same gap. Both are heavily right-skewed, so medians, log scales and rank-based tests are used below.

In [ ]:
print("Skewness  LOS:", round(stats.skew(df.los),1), "| charges:", round(stats.skew(df.total_charges),1))
px.histogram(np.log10(df.total_charges.sample(min(len(df), 50000), random_state=1)), nbins=60,
             title="log10(total charges): much closer to symmetric", labels={"value":"log10(charges)"})

## 2. Q1: Which diagnoses are linked to long stays?

In [ ]:
dx = (df.groupby(["dx_code","dx_description","dx_category"], observed=True)
        .agg(cases=("los","size"), avg_los=("los","mean"), median_los=("los","median"), avg_charge=("total_charges","mean"))
        .query("cases >= 500").sort_values("avg_los", ascending=False))
dx["est_bed_days"] = dx.cases * SCALE * dx.avg_los
display(dx.head(10)); display(dx.sort_values("est_bed_days", ascending=False).head(5))

## 3. Q2: Which age groups incur the highest charges?

In [ ]:
age = df.groupby("age_group", observed=True).agg(cases=("los","size"), avg_charge=("total_charges","mean"),
        median_charge=("total_charges","median"), total_bn=("total_charges", lambda s: s.sum()*SCALE/1e9))
age["share_of_charges_%"] = age.total_bn / age.total_bn.sum() * 100
age

## 4. Q3: How does severity affect cost?

In [ ]:
sev = df.groupby("severity", observed=True).agg(cases=("los","size"), avg_los=("los","mean"),
        median_charge=("total_charges","median"), avg_charge=("total_charges","mean"))
sev["median_x_vs_minor"] = sev.median_charge / sev.median_charge.iloc[0]
sev["share_discharges_%"] = sev.cases / sev.cases.sum() * 100
sev["share_charges_%"] = df.groupby("severity", observed=True).total_charges.sum() / df.total_charges.sum() * 100
sev

In [ ]:
groups = [g.total_charges.values for _, g in df.sample(min(len(df), 60000), random_state=1).groupby("severity", observed=True)]
H, p = stats.kruskal(*groups); print(f"Kruskal-Wallis H = {H:,.0f}, p = {p:.2e}  -> charges differ significantly across severity levels")

## 5. Q4: Are some counties slower? (pivot table)

In [ ]:
cty = df[df.county != "Unknown"].groupby("county", observed=True).agg(cases=("los","size"), avg_los=("los","mean"), avg_charge=("total_charges","mean"))
big = cty[cty.cases >= 10000].sort_values("avg_los", ascending=False); display(big)
# severity-adjusted view: average stay by county x severity
display(df[df.county.isin(big.index)].pivot_table(index="county", columns="severity", values="los", aggfunc="mean", observed=True))

**Why the pivot matters:** if a county's stay is longer *within every severity band*, the gap is not just a sicker patient mix.

## 6. Q5: How do payment types affect billing?

In [ ]:
pay = df.groupby("payer_primary", observed=True).agg(cases=("los","size"), avg_los=("los","mean"), avg_charge=("total_charges","mean"),
        avg_cost=("total_costs","mean"), charges=("total_charges","sum"), costs=("total_costs","sum"))
pay["charge_to_cost"] = pay.charges / pay.costs
pay["share_patients_%"] = pay.cases / pay.cases.sum() * 100
pay.drop(columns=["charges","costs"]).sort_values("avg_charge", ascending=False)

## 7. Q6: Does route of admission matter?

In [ ]:
df.pivot_table(index="admission_type", columns="med_surg", values=["los","total_charges"], aggfunc="mean", observed=True)

## 8. Q7: High-utilisation facilities

In [ ]:
fac = (df[~df.facility_redacted].groupby("facility", observed=True)
         .agg(cases=("los","size"), avg_los=("los","mean"), avg_charge=("total_charges","mean"), avg_cost=("total_costs","mean"))
         .query("cases >= 1000").sort_values("cases", ascending=False))
display(fac.head(10))
print("Spread of avg charge among top 10 busiest:", round(fac.head(10).avg_charge.max()/fac.head(10).avg_charge.min(), 2), "x")

## 9. Correlation and regression

In [ ]:
s = df.sample(min(len(df), 60000), random_state=1)
num = s[["los","total_charges","total_costs","severity_code","n_payers"]]
display(num.corr(method="spearman").round(2))
slope, intercept, r, p, se = stats.linregress(np.log10(s.los), np.log10(s.total_charges))
print(f"log-log fit: charges ~ stay^{slope:.2f}, R^2 = {r**2:.2f}")

## 10. Conclusions
1. Psychiatric and neurological diagnoses have the longest stays; septicemia carries the largest bed-day and charge burden.
2. Patients 50+ generate about 69% of all charges.
3. Severity is the dominant cost driver (median charge up 4.7x from Minor to Extreme).
4. Medicare and Medicaid cover about 70% of patients; hospitals bill roughly 3.3x estimated cost.
5. Facility and county differences should be interpreted after adjusting for severity mix.

These findings feed the interactive dashboard in `app.py`.